## Introduction to Array and DForm

We introduce a new way of using arrays in our python environmemt.
First of all, we should understand what is missing now in numpy's ndarray.
1. We can't save metadata for the array - for example color,units, position of color axis etc
2. Changing between 2 data representations is dificult - for example changing from rgb with range 0-1 to gray with range 0-255 requires some lines of code

We introduce DataForm, or DForm, which is an extension of dtype of ndarray.
DForm is a class which includes two main features - inclusion of metadata and transformations between DForms

We also intruduce the metaclass FormArray, which help you create new classes for array-like data with metadata through dform.



In [21]:
from inu.utils.array import *
import numpy as np

### Hello Array!

Let's see an example

In [2]:
dform = DForm(color='RGB', dtype=np.uint8)

In [3]:
print(dform,'\n', 'range :', dform.range, 'color :', dform.color)

Ⓕu1⎣0➔255⌉ 
 range : 255 color : RGB


<font size="3">We can see that the dform got range from 0 to 255, as expected from uint8 dtype</font>

<font size="3">Let's create new class of arrays with the dform</font>

In [4]:
Array_RGB = FormArray(dform)

array = Array_RGB(np.random.randint(dform.min, dform.max, (2, 4, 3)))
array

Ⓕu1⎣0➔255⌉〚2×4×3〛 ⧮⟪10⦙252⟫:
[[[206 162 126]
  [182  80 185]
  [ 98  52  67]
  [113 185  13]]

 [[133  10 115]
  [ 17  30 252]
  [ 68 115 246]
  [ 58 142 155]]]

<font size="3">The first benefits of the FormArray class constructor is that we get extra information when printing the array, which for big arrays are more important

In [5]:
# include mean and std in printing
FormArray.global_print_options(norm=True)

array = Array_RGB(np.random.randint(dform.min, dform.max, (100, 200, 3)))
array

Ⓕu1⎣0➔255⌉〚100×200×3〛 ⧮⟪0⦙254⟫ <127>σ73.5

<font size="3">For example, here for a bigger array there is less importance for the data because we can only see the borders.
<font size="3">But we have more important info - the size, the range ( of the array, not the limits) and the mean and STD

<font size="3">If you want to see the data as it was in ndarray, use the method:

In [6]:
array.ndarray

array([[[  9, 170, 163],
        [106,  54, 118],
        [ 78, 188, 120],
        ...,
        [ 40, 249, 104],
        [ 36,  24, 211],
        [ 20, 164,  13]],

       [[125,  37,  69],
        [128,  32, 128],
        [162, 184,  52],
        ...,
        [165, 223,  96],
        [ 34,  14,  45],
        [ 47,  15, 158]],

       [[162,  25,  47],
        [185, 132, 175],
        [231, 136, 168],
        ...,
        [211, 174,  23],
        [161, 154,  25],
        [ 77, 119,  52]],

       ...,

       [[163,  19,  88],
        [ 86,  33, 118],
        [241, 212,  86],
        ...,
        [252, 113,  58],
        [238, 138, 122],
        [251, 165, 246]],

       [[247, 242,  99],
        [ 40,  83, 244],
        [ 13, 234, 157],
        ...,
        [181, 116,  24],
        [203, 168, 169],
        [  4,  43,  84]],

       [[161, 146,  42],
        [119,  53,  85],
        [180, 102,  39],
        ...,
        [120, 151,  85],
        [217,  93, 178],
        [162, 100, 112]]

### DForm in more details

As seen above, DForm is an extansion to dtype and wants to give important info to the user, and use it to do fast transformations between data represantations

DForm includes metadata you can set and use later with your data.

DForm attributes are:
1. name
2. dtype
3. range - min and max
4. normalization - mean and std
5. color - RGB or GRAY
6. color axis position
7. units 
8. kind

In [7]:
dform = DForm(name= 'new_dform', min= 0, max = 1000, dtype=np.float32, color='RGB',avr=1, std=2)
for attr, value in dform.items(): print(attr, value)

name new_dform
dtype float32
min 0.0
max 1000.0
avr [1]
std [2]
color RGB
cax -1
units None
kind Kind.IMG


<font size="3">DForm can be created in both usuall fashion with parameters, or with a name which would be parsed and create every parameter included in the name, using a pattern

In [8]:
dform1 = DForm(name= 'my_dform', min= 0 , max = 1000, dtype=np.float32, color='GRAY')
dform1

Ⓕmy_dformf4⎣0➔1000⌉

In [9]:
dform2 = DForm('GR0:1000f4')
dform2

ⒻGR0:1000f4f4⎣0➔1000⌉

In [10]:
dform1 == dform2

True

In [11]:
try:

    dform2 = DForm('TR0:1000f4')

except NameError as exp:

    print(exp)

Unknown DForm 'TR0:1000f4'


<font size="3">All dforms saved in a registered list of the DForm class

In [12]:
DForm.list_registered()

  0. ImgNet              : ⒻImgNetu1<124, 116, 104>σ58.4, 57.1, 57.4
  1. new_dform           : Ⓕnew_dformf4⎣0➔1000⌉<1>σ2
  2. my_dform            : Ⓕmy_dformf4⎣0➔1000⌉
  3. GR0:1000f4          : ⒻGR0:1000f4f4⎣0➔1000⌉


<font size="3">More options of creating a dform includes:
    
    1. from_data - creating a new dform from an array
    2. from_merge - creating a new dform from a specific one, with updated fields.

In [13]:
dform_source = DForm(color='RGB', dtype=np.float64, min=0, max=1)
Array_RGB = FormArray(dform_source)
array = Array_RGB(np.random.rand(2, 4, 3))

dform_out = DForm.from_data(array)
for attr, value in dform_out.items(): print(attr, value)

name None
dtype float64
min None
max None
avr None
std None
color None
cax None
units None
kind None


In [14]:
dform = DForm.from_merge(dform_source, color='GRAY', dtype=int)
for attr, value in dform.items(): print(attr, value)

name None
dtype int64
min 0
max 1
avr None
std None
color GRAY
cax -1
units None
kind Kind.IMG


### Creating new Array class

<font size="3">The use case emerges when we want different types of data containers, which differ not only by data types, but differ with color, ranges, specific normalization that they came from, and more metadata we want to add.

<font size="3">Create a new data container type by using FormArray metaclass with a specific dform.

In [22]:
dform = DForm('GR0:255f8')
Array_Gray = FormArray(dform)

### Dealing with arrays

<font size="3"> There are 3 basic ways to create a newer array instance with metadata:

#### 1. Using Array Base Class

In [38]:
data = np.random.randint(500, 1000, (4,4))
a = Array(data)
a

☉f4〚4×4〛 ⧮⟪5.1e+02⦙1e+03⟫ <799>σ152.8982391357422:
[[780. 694. 889. 590.]
 [690. 510. 910. 769.]
 [995. 536. 997. 843.]
 [778. 931. 936. 929.]]

In [39]:
a = Array(data, DForm('GR0:255f8'))
a

ⒻGR0:255f8f8⎣0➔255⌉〚4×4〛 ⧮⟪5.1e+02⦙1e+03⟫ <799>σ152.89823770648894:
[[780. 694. 889. 590.]
 [690. 510. 910. 769.]
 [995. 536. 997. 843.]
 [778. 931. 936. 929.]]

Notice that the class of the instance created is not "Array"!

In [40]:
type(a)

ArrayGR0:255f8

<font size="3">By default the array instance created **does not change the data** to fit the dform. You can see the values are not in the range 0->255

<font size="3"> However we can turn on the trans paramter to do the transformation to the new dform.

In [41]:
a = Array(data, dform = DForm('GR0:255f8'), trans=True)
# Notice the range is 0 to 255

assert (a < 255).all()
assert (a > 0).all()
a

AssertionError: 

<font size="3"> We can also include specific keyword arguments to be merged to the dform:

In [28]:
a = Array(data, dform = DForm('GR0:255f8'), kind='disp')
a

TypeError: Array.__new__() got an unexpected keyword argument 'kind'

#### 2. Using "asform" or "transform" methods of an existing array instance (not to be confused with ndarray).

<font size="3">Method "asform" doesn't change the data, and only creates new instance with the same data and the new dform

In [29]:
a = Array(data, dform = DForm('GR0:255f8'))
b = a.asform( DForm('GR0:1f4'))
assert (a == b).all()
assert type(b) != type(a)

<font size="3">We can add more kws to the new dform

In [30]:
b = a.asform( DForm('GR0:1f4'), kind='disp')
b.dform

/home/avive/code/algodev/inu/utils/array.py:518: UserWarning: DForm('GR0:1f4') overrides registered GR0:1f4
  warnings.warn(f"{cls.__name__}('{dform.name}') overrides registered {found.name}")


KeyError: "DForm has no attribute: {'kind'}"

<font size="3">3.Method "transform" change the data according to the new dform.

In [31]:
a = Array(data, dform = DForm('GR0:255f8'))
b = a.transform( DForm('GR0:1f4'))
assert (a != b).all()
b

ⒻGR0:1f4f4⎣0➔1⌉〚4×4〛 ⧮⟪2.2⦙3.7⟫ <3.03>σ0.5691425204277039:
[[2.561 3.357 3.282 3.573]
 [2.655 2.157 3.725 2.349]
 [2.761 3.69  3.329 3.169]
 [2.29  2.227 3.639 3.714]]

<font size="3">3. Using existing Array like class as the constructor

#### More options to create array instance

<font size="3">You can skip the creation of a data container and create an instance immidietly using form_array function, which is created using the same datacontainer. It as an alias to a method of the metaclass: FormArray.from_array

<font size="3">It's important to understand that **dform defines an Array class**, which means that: **same dform --> same Array class**

In [23]:
array = form_array(np.zeros((1,1,3)), dform=dform)
array_2 = FormArray.from_array(np.zeros((1,1,3)), dform=dform)

type(array) == type(array_2) == Array_Gray

True

### FormArray advanced methods

In [24]:
dform = DForm(color='RGB', dtype=np.uint8)
Array_RGB = FormArray(dform,kind='disp')

TypeError: FormArray.__new__() got an unexpected keyword argument 'kind'

<font size="3">FormArray metaclass allows also to change global printing options for all Array classes created.

<font size="3">For example - changing the maximum rows and columns to show, precision, showing stats etc

In [25]:
FormArray.global_print_options(rows=8, cols=8, stats=True)
array = form_array(np.random.randint(dform.min, dform.max, (15,15)))
array
type(array)

Array_i8

In [26]:
FormArray.global_print_options(rows=16, cols=16, stats=False)
array

☉i8〚15×15〛:
[[220 240 216 203   2  37  42 239   3 113  17  68 236 166  68]
 [127 232 245  70  64 254 130 151 207 249 218 225 208 210 224]
 [179  84  30  34 112  57 189   5 205 201 203 254  88 163 121]
 [ 70  24  71  96  95  90 182 135 124 248  73 224 198  88  74]
 [203 145 244  58  81 206  50 169  84 104 241 162 225 202  34]
 [221 184 177  40 212  17   3  26 166   2  40 124 193  56 120]
 [ 74  72 102 126 202  37  97 246  69 105  42 180  44 105 200]
 [137 249 183 239 150 235 186 224   9 116 134 232 172 219   9]
 [ 34   0  65  56 172 122 138  74 115  42 146  46  27  17 200]
 [ 62 194 222 238 131 209 182 106 254 174  12 130 218  92  60]
 [ 33  15 206 214 199 162 229  10   2 151 150  91 217  30  21]
 [160 106 238 231 185 163  90 228  12 151  49 150  82  70  48]
 [247 102  30 190  18 240 229 215 236 180  99 140 132  83  12]
 [148 161 222 251 153 241  21 174   4 183  65  15  82 144  32]
 [ 86 185 167 133  34 233   9 109 105 114 193 181 110  59  20]]

You can always see what are the current options by calling the method without parameters

In [27]:
FormArray.global_print_options()

{'edgeitems': 3,
 'threshold': 1000,
 'floatmode': 'maxprec',
 'precision': 3,
 'suppress': False,
 'linewidth': 75,
 'nanstr': 'nan',
 'infstr': 'inf',
 'sign': '-',
 'formatter': None,
 'legacy': False,
 'rows': 16,
 'cols': 16,
 'rng': True,
 'norm': True,
 'nans': True,
 'stats': False,
 'info': True}

You can also use print_options method of an Array class  to change printing options of a specific Array class

In [28]:
Array_Gray.print_options(rows=8,cols=8)
Array_Gray.print_options()

{'rows': 8,
 'cols': 8,
 'precision': 3,
 'rng': True,
 'norm': True,
 'nans': True,
 'stats': False,
 'info': True}

In [29]:
FormArray.global_print_options()['rows'] == Array_Gray.print_options()['rows']

False

### Array Base Class in more details

In [30]:
dform = DForm('GR0:255f8')
a = Array(data,dform = DForm('GR0:255f8'), trans=True)
# Barray = FormArray(DForm('GR0:255f8'))
# a = Barray((np.random.randint(500, 1000, (4,4))),DForm('GR0:255f8'))
import sys, inspect
clsmembers = inspect.getmembers(sys.modules[__name__], inspect.isclass)
clsmembers
type(a)

ArrayGR0:255f8

### Transformations

The second benefit of using DForm and Array classes are transformation between two types of Arrays (by transforming between the dforms of each)

#### asform()

The first option of transformation is using asform method, which changes the array's (instance) dform, but not changing the data itself!

In [31]:
dform = DForm('GR0:255f8')
array = FormArray(dform)(np.random.randint(dform.min, dform.max, (4,4)))
array

ⒻGR0:255f8f8⎣0➔255⌉〚4×4〛:
[[  1. 213.  54. 252.]
 [178. 102. 198. 208.]
 [100.  39. 182. 202.]
 [ 47. 109.  59. 205.]]

In [32]:
new_array = array.asform(DForm('GR0:1f8'))
new_array

array([[  1., 213.,  54., 252.],
       [178., 102., 198., 208.],
       [100.,  39., 182., 202.],
       [ 47., 109.,  59., 205.]])

And as we can see, the range is not the same as the data. Meaning it's YOUR responsibility to change only the dform of the array.

#### transform()

The second option is full transformation of the data using the transform method which changes the array's (instance) dform and updated the data according to the dform.

In [33]:
new_array = array.transform(DForm('GR0:1f8')) 
new_array

/home/avive/code/algodev/inu/utils/array.py:517: UserWarning: DForm('GR0:1f8') overrides registered GR0:1f8
  warnings.warn(f"{cls.__name__}('{dform.name}') overrides registered {found.name}")


ⒻGR0:1f8f8⎣0➔1⌉〚4×4〛:
[[0.004 0.835 0.212 0.988]
 [0.698 0.4   0.776 0.816]
 [0.392 0.153 0.714 0.792]
 [0.184 0.427 0.231 0.804]]

And we can see that the data updated from range 0->255 to 0->1

There are some transformations supported:

1. range -> range
2. dtype -> dtype (casting, and rounding if shrinking the size)
3. color -> color
4. normalization: (avr, std) -> (avr, std)

#### RGB to gray

In [34]:
dform = DForm('CR0:255f8')
array_rgb = FormArray(dform)(np.random.randint(dform.min, dform.max, (20,20,3)))
array_rgb

ⒻCR0:255f8f8⎣0➔255⌉〚20×20×3〛

In [35]:
array_gray = array_rgb.transform(DForm('GR0:255f8'))
array_gray

ⒻGR0:255f8f8⎣0➔255⌉〚20×20〛